# POS Tagging, Morphology, Regular Expressions and Finite Automata

Day 1, Session 3. Labelling words, breaking them into parts, and matching patterns with machines

Run this in Google Colab (or any Jupyter setup). Run the cells top to bottom.

## Setup

In [ ]:
# one time installs, colab already has most of these
# nltk  -> tokenizer, pos tagger, simple grammar parser
# spacy -> pos tags and morphology in one call
%pip install -q nltk spacy
!python -m spacy download en_core_web_sm

In [ ]:
import re
import nltk
import spacy

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

nlp = spacy.load("en_core_web_sm")

## 1. Part-of-speech tagging

Same word, different tag depending on context.

In [ ]:
s = "She will fish for fish"
print(nltk.pos_tag(nltk.word_tokenize(s)))
print([(t.text, t.pos_) for t in nlp(s)])

In [ ]:
# taggers make mistakes, compare them on a few sentences
tests = ["I read a book", "I will book a flight", "Time flies like an arrow"]
for s in tests:
    print(s)
    print("  nltk :", nltk.pos_tag(nltk.word_tokenize(s)))
    print("  spacy:", [(t.text, t.pos_) for t in nlp(s)])

In [ ]:
# count tag frequencies in a short paragraph
para = "The quick brown fox jumps over the lazy dog. The dog barked loudly and the fox ran away."
tags = [tag for _, tag in nltk.pos_tag(nltk.word_tokenize(para))]
print(nltk.FreqDist(tags).most_common())

## 2. Morphology

In [ ]:
# spacy stores grammar info in token.morph
doc = nlp("She walked three dogs")
for t in doc:
    print(t.text, t.lemma_, t.morph)

In [ ]:
# a tiny affix splitter, only for teaching
prefixes = ["un", "re", "dis"]
suffixes = ["ness", "ing", "ed", "ly", "s"]

def split_word(word):
    prefix, suffix = "", ""
    for p in prefixes:
        if word.startswith(p) and len(word) > len(p) + 2:
            prefix, word = p, word[len(p):]
            break
    for suf in suffixes:
        if word.endswith(suf) and len(word) > len(suf) + 2:
            suffix, word = suf, word[:-len(suf)]
            break
    return prefix, word, suffix

for w in ["unhappiness", "rewalked", "quickly", "dogs", "mice"]:
    print(w, split_word(w))

# note: unhappiness gives 'happi' as the root, spelling changes break simple rules
# mice has no suffix at all, that is why real morphology needs dictionaries

## 3. Regular expressions

In [ ]:
msg = "Mail rao@nlp.org or call 98765 43210 on 12/10/2026 #NLP"
print(re.findall(r"[\w.]+@[\w.]+\.\w+", msg))
print(re.findall(r"\b[6-9]\d{4}\s?\d{5}\b", msg))
print(re.findall(r"\d{2}/\d{2}/\d{4}", msg))
print(re.findall(r"#\w+", msg))

In [ ]:
# fullmatch checks the whole string, search finds a pattern anywhere
def is_phone(s):
    return bool(re.fullmatch(r"[6-9]\d{9}", s))

for s in ["9876543210", "1234567890", "98765"]:
    print(s, is_phone(s))

# sub replaces matches, handy for masking
print(re.sub(r"\d{5}\s?\d{5}", "XXXXXXXXXX", "call 98765 43210 now"))

In [ ]:
# groups let you pull out parts of a match
m = re.search(r"(\d{2})/(\d{2})/(\d{4})", "workshop on 12/10/2026")
print(m.groups())
print("day", m.group(1), "month", m.group(2), "year", m.group(3))

# greedy vs lazy: .* takes as much as it can, .*? takes as little
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.*>", html))
print(re.findall(r"<.*?>", html))

## 4. Finite automata

A DFA for binary strings ending in 01, written as a table.

In [ ]:
delta = {("q0", "0"): "q1", ("q0", "1"): "q0",
         ("q1", "0"): "q1", ("q1", "1"): "q2",
         ("q2", "0"): "q1", ("q2", "1"): "q0"}

def accepts(s, trace=False):
    state = "q0"
    path = [state]
    for ch in s:
        state = delta[(state, ch)]
        path.append(state)
    if trace:
        print(" -> ".join(path))
    return state == "q2"

for s in ["01", "1101", "100", "0101", "10"]:
    print(s, accepts(s, trace=True))

In [ ]:
# the same language as a regex, they agree on every string
import itertools
pattern = re.compile(r"[01]*01")
for n in range(1, 5):
    for bits in itertools.product("01", repeat=n):
        s = "".join(bits)
        assert accepts(s) == bool(pattern.fullmatch(s))
print("dfa and regex agree on all binary strings up to length 4")

## 5. Limits of regular languages

In [ ]:
def balanced(s):
    depth = 0                      # this counter is the memory a dfa does not have
    for ch in s:
        depth += 1 if ch == "(" else -1
        if depth < 0:
            return False
    return depth == 0

for t in ["(())", "(()", "()()", ")("]:
    print(t, "regex says", bool(re.fullmatch(r"\(+\)+", t)), "| counter says", balanced(t))

### Bonus: a context-free grammar for 'the cat eats food'

This is the case study from the syllabus. A grammar gives us a tree, which regular expressions cannot.

In [ ]:
grammar = nltk.CFG.fromstring("""
S  -> NP VP
NP -> Det N | N
VP -> V NP
Det -> 'the'
N  -> 'cat' | 'food'
V  -> 'eats'
""")
parser = nltk.ChartParser(grammar)

for tree in parser.parse("the cat eats food".split()):
    print(tree)
    tree.pretty_print()

In [ ]:
# a sentence the grammar does not allow gives no parse
words = "cat the eats food".split()
print(list(parser.parse(words)))

## Practice

1. (Easy) Tag 'Time flies like an arrow' with both libraries. Do they agree? Which tagging do you find more sensible?

2. (Medium) Write a regex that extracts all prices from 'Rs 500, Rs. 1,200 and Rs.75.50'. Return only the numbers.

3. (Challenge) Build a DFA, as a transition table in Python, that accepts binary strings with an even number of 0s. Test it on five strings.

In [ ]:
# your practice code here